# BIRD Mini-Dev: Schema Pruning trên Kaggle

Notebook có hai giai đoạn: (1) retrieval trên 500 câu chính thức, không cần GPU/SQLite; (2) đối chiếu full/BM25/hybrid khi đã gắn đúng SQLite và model.

Chọn accelerator có sẵn trong Notebook Settings. Bật Internet nếu tải dependency/model/data; phiên offline cần gắn sẵn các artifact. Mọi kết quả được ghi dưới `/kaggle/working`.

Gold SQL chỉ được dùng ở evaluator. Nhãn schema suy ra từ gold SQL; đây không phải annotation schema linking chính thức. EX của dự án dùng bộ so sánh riêng, cần scorer chính thức để báo cáo BIRD EX/R-VES.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Sửa đường dẫn này theo dataset chứa source code bạn đã thêm vào Kaggle.
REPO_INPUT = Path("/kaggle/input/slm-prj/SLM-PRJ")
WORK = Path("/kaggle/working/SLM-PRJ")
assert (REPO_INPUT / "core/pipeline.py").is_file(), f"Sửa REPO_INPUT: {REPO_INPUT}"
shutil.copytree(REPO_INPUT, WORK, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", ".cache", "results", "artifacts"))
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLCONFIGDIR"] = "/kaggle/working/matplotlib-cache"
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True, cwd=WORK)
print(WORK)


In [ ]:
# Kaggle thường có sẵn PyTorch; giữ build GPU của môi trường.
# Nếu Internet tắt, chuẩn bị wheelhouse và thay lệnh này bằng pip --no-index --find-links.
run("-m", "pip", "install", "-r", "requirements.txt", "-r", "requirements-model.txt")
run("-m", "pytest", "-q")


## 1. Tải metadata chính thức đã ghim phiên bản

Ba tệp khoảng 538 KiB, gồm 500 câu, schema 11 DB và manifest. Script kiểm tra SHA-256. Không tải SQLite hoặc trọng số model.

Nếu offline, thay `DATA` bằng thư mục metadata đã gắn sẵn và bỏ lệnh tải. Dataset revision được ghim trong script; không trộn với archive 2025 hoặc bộ dev khác.


In [ ]:
DATA = WORK / "data/bird_minidev"
run("scripts/download_bird_minidev.py", "--output", DATA)
QUESTIONS = DATA / "mini_dev_sqlite.jsonl"
SCHEMAS = DATA / "schemas.json"
print(json.loads((DATA / "provenance.json").read_text())["revision"])


In [ ]:
import pandas as pd
# Baseline và BM25 chỉ chạy CPU. Token mặc định là proxy regex, được gắn nhãn rõ.
from evaluation.evaluate_schema_retrieval import evaluate as evaluate_retrieval
retrieval_reports = []
for mode, tables, columns in [("full", 4, 8), ("bm25", 4, 8), ("bm25", 6, 16), ("bm25", 8, 24), ("bm25", 8, 48)]:
    label = "full" if mode == "full" else f"{mode}_t{tables}_c{columns}"
    report = evaluate_retrieval(QUESTIONS, SCHEMAS, WORK / "results/retrieval" / label,
                               mode=mode, top_k_tables=tables, top_k_columns=columns)
    retrieval_reports.append({"label": label, **report["summary"]})
pd.DataFrame(retrieval_reports)[["label", "questions", "label_errors", "column_recall_macro",
    "all_references_covered", "prompt_token_reduction_micro", "retrieval_ms_p95"]]


## 2. Đếm token thật và thử hybrid (tùy chọn)

Dùng cùng tokenizer với mô hình đánh giá. Token ở bước retrieval là văn bản prompt thô; backend khi sinh SQL ghi riêng số token sau chat template.

Hybrid tải encoder riêng; cache vector lưu dưới `/kaggle/working`. Với 11 DB nhỏ, cosine trên ma trận là đủ, chưa cần dịch vụ vector database. Chạy lượt đầu xây index và lượt thứ hai để so sánh cache; giữ log cả hai.


In [ ]:
import hashlib
from transformers import AutoTokenizer
from core.config import load_config
RUN_SEMANTIC = False  # Bật khi muốn chạy thêm hybrid.
MODEL_ID = "cycloneboy/SLM-SQL-1.5B"  # Hoặc đường dẫn model đã gắn sẵn.
MODEL_REVISION = "2cc2df91aacafdec3fef3733ac45bd53b57d54cf"  # Đổi cùng MODEL_ID nếu dùng model khác.
ENCODER_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
ALLOW_DOWNLOADS = True  # False khi dùng cache/model offline.
config = load_config()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, local_files_only=not ALLOW_DOWNLOADS)
tokenizer_state = {"vocab": tokenizer.get_vocab(), "special_tokens": tokenizer.special_tokens_map,
    "chat_template": tokenizer.chat_template,
    "backend": tokenizer.backend_tokenizer.to_str() if hasattr(tokenizer, "backend_tokenizer") else None}
TOKENIZER_SHA256 = hashlib.sha256(json.dumps(tokenizer_state, sort_keys=True, default=str).encode()).hexdigest()
TOKENIZER_TAG = f"{MODEL_ID}@{MODEL_REVISION}#{TOKENIZER_SHA256}"
for mode in (["full", "bm25", "hybrid"] if RUN_SEMANTIC else ["full", "bm25"]):
    if mode == "hybrid":
        run("-m", "pip", "install", "-r", "requirements-retrieval.txt")
    report = evaluate_retrieval(QUESTIONS, SCHEMAS, WORK / "results/retrieval_tokens" / mode,
        mode=mode, top_k_tables=8, top_k_columns=48, tokenizer=tokenizer, tokenizer_name=TOKENIZER_TAG,
        retriever_options={"local_files_only": not ALLOW_DOWNLOADS,
            "embedding_revision": ENCODER_REVISION, "device": "cpu", "cache_dir": str(WORK / ".cache/schema_embeddings")})
    print(mode, report["summary"])


## 3. Gắn SQLite cùng phiên bản và chuyển dữ liệu

Thêm package SQLite chính thức từ hướng dẫn BIRD, giải nén thành `<DB_ROOT>/<db_id>/<db_id>.sqlite`. Chỉ chạy tiếp khi đã đối chiếu phiên bản SQLite với manifest/questions. Metadata mới và DB archive cũ có thể không khớp; không coi lỗi gold SQL là lỗi model.

Để chạy BIRD dev, thay QUESTIONS bằng `dev.json` tương ứng và dùng đúng `dev_databases`; phần retrieval hỗ trợ `dev_tables.json` theo định dạng BIRD/Spider.


In [ ]:
RUN_GENERATION = False  # Bật sau khi đã gắn SQLite và model.
DB_ROOT = Path("/kaggle/input/bird-sqlite/dev_databases")  # Sửa theo dataset đã gắn.
DATASET = WORK / "data/benchmarks/bird_eval.json"
if RUN_GENERATION:
    run("scripts/import_bird.py", "--questions", QUESTIONS, "--database-root", DB_ROOT, "--output", DATASET)


## 4. So sánh cùng model, thay duy nhất schema

`LIMIT=10`, `REPEATS=1` là smoke test. Chạy chính thức dùng `LIMIT=None`; tăng `REPEATS=3` để xoay thứ tự các arm và quan sát nhiễu latency. Mỗi repeat dùng cùng seed để cô lập pruning; đây không phải nhiều seed model độc lập. Không gộp các repeat thành nhiều câu hỏi độc lập.

Notebook ghim model snapshot, hash trọng số/SQLite, ghi hardware và dependency vào `runtime_manifest.json` cho từng arm. Hash tệp lớn có thể mất thời gian. Greedy, batch 1, correction off, sample rows 0. Warm-up GPU riêng cho mỗi arm được loại khỏi báo cáo; selector đánh giá mới vẫn tính chi phí khởi tạo/first access theo DB. Cache vector trên disk có thể dùng lại giữa các repeat; giữ chính sách này khi đọc timing.

Nếu gold SQL vượt giới hạn execution, điều chỉnh cùng cấu hình cho tất cả arm. Chốt ngân sách trước tập đánh giá cuối. Xem `docs/EVALUATION_A_TO_Z.md` để phân biệt raw prompt token, token sau chat template, EX nội bộ, EX chính thức, và giới hạn thống kê.


In [ ]:
if RUN_GENERATION:
    import copy
    import gc
    import hashlib
    import importlib.metadata
    import platform
    import torch
    from huggingface_hub import snapshot_download
    from core.model_backend import create_backend
    from evaluation.evaluate import evaluate as evaluate_sql
    from core.pipeline import Pipeline

    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings"
    LIMIT = 10  # None = toàn bộ 500 câu; không dùng smoke test để báo benchmark.
    REPEATS = 1  # 3 để quan sát ảnh hưởng thứ tự/nhiễu runtime.
    assert REPEATS >= 1
    model_snapshot = (str(Path(MODEL_ID).resolve()) if Path(MODEL_ID).is_dir() else
        snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_files_only=not ALLOW_DOWNLOADS))
    config["model"].update(backend="huggingface", model_id=model_snapshot, device="cuda", local_files_only=True)
    config["sampling"]["mode"] = "greedy"
    config["generation"].update(batch_size=1, max_new_tokens=512)
    config["correction"]["enabled"] = False
    config["schema"]["sample_rows"] = 0
    config["retrieval"].update(top_k_tables=8, top_k_columns=48,
        embedding_revision=ENCODER_REVISION, local_files_only=not ALLOW_DOWNLOADS,
        cache_dir=str(WORK / ".cache/schema_embeddings"))
    config["execution"].update(timeout_seconds=30, max_steps=50_000_000,
        max_result_rows=100_000, max_result_bytes=64*1024*1024)
    items = json.loads(DATASET.read_text())
    items = items if LIMIT is None else items[:LIMIT]
    assert items, "Dataset rỗng"
    EVAL_DATASET = DATASET.with_name("bird_eval_subset.json")
    EVAL_DATASET.write_text(json.dumps(items, ensure_ascii=False, indent=2))
    first = items[0]
    first_db = (EVAL_DATASET.parent / first["db_path"]).resolve()
    # Khi rerun sau smoke, giải phóng model cũ trước khi nạp model mới.
    if "backend" in globals():
        globals().pop("backend")
        gc.collect()
        torch.cuda.empty_cache()
    backend = create_backend(config)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()

    snapshot = Path(model_snapshot)
    weight_hashes = {str(path.relative_to(snapshot)): sha256_file(path)
        for path in sorted(snapshot.rglob("*")) if path.is_file()}
    db_paths = {item["db_id"]: (EVAL_DATASET.parent / item["db_path"]).resolve() for item in items}
    manifest = {
        "model_id": MODEL_ID, "declared_revision": MODEL_REVISION,
        "resolved_model_revision": getattr(backend.model.config, "_commit_hash", None),
        "model_files_sha256": weight_hashes,
        "database_sha256": {db_id: sha256_file(path) for db_id, path in sorted(db_paths.items())},
        "dataset_sha256": sha256_file(EVAL_DATASET),
        "tokenizer_tag": TOKENIZER_TAG,
        "parameter_dtype": str(next(backend.model.parameters()).dtype),
        "hf_device_map": {str(k): str(v) for k, v in getattr(backend.model, "hf_device_map", {}).items()},
        "is_loaded_in_4bit": bool(getattr(backend.model, "is_loaded_in_4bit", False)),
        "is_loaded_in_8bit": bool(getattr(backend.model, "is_loaded_in_8bit", False)),
        "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
        "packages": {name: importlib.metadata.version(name) for name in ["transformers", "numpy", "sqlglot"]},
        "gpus": [{"index": i, "name": torch.cuda.get_device_name(i),
                  "total_memory_bytes": torch.cuda.get_device_properties(i).total_memory}
                 for i in range(torch.cuda.device_count())],
        "cache_policy": "GPU warmup per arm; fresh evaluation selector; disk embeddings reusable",
    }
    GENERATION_ROOT = WORK / "results/generation"
    ARMS = ["full", "bm25", "hybrid"] if RUN_SEMANTIC else ["full", "bm25"]
    summaries = []
    for repeat in range(REPEATS):
        arm_order = ARMS[repeat % len(ARMS):] + ARMS[:repeat % len(ARMS)]
        repeat_root = GENERATION_ROOT / f"repeat_{repeat:02d}"
        repeat_root.mkdir(parents=True, exist_ok=True)
        (repeat_root / "run_order.json").write_text(json.dumps(arm_order))
        for mode in arm_order:
            arm = copy.deepcopy(config)
            arm["validation"]["schema_selection"] = mode != "full"
            if mode != "full":
                arm["retrieval"]["mode"] = mode
            Pipeline(arm, backend).run(first_db, first["question"], first.get("evidence", ""))
            output = repeat_root / mode
            summary = evaluate_sql(arm, EVAL_DATASET, first_db, output, backend=backend, profile_cuda=True)
            (output / "runtime_manifest.json").write_text(json.dumps(manifest, indent=2))
            summaries.append({"repeat": repeat, **summary})
    display(pd.DataFrame(summaries)[["repeat", "schema_mode", "questions", "execution_accuracy",
        "average_latency_ms", "mean_model_input_tokens_per_call", "context_overflows",
        "peak_cuda_allocated_mb_sum_device_peaks"]])
    with (WORK / "results/pip-freeze.txt").open("w") as handle:
        subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)
    print("Resolved model snapshot:", model_snapshot)


## 5. Vẽ biểu đồ và so sánh theo từng câu

Cell đầu vẽ số liệu retrieval đã đo và lưu trong repo. Cell thứ hai dùng chính output generation của phiên này; thiếu CUDA/token sẽ bỏ biểu đồ tương ứng và ghi `plot_notes.json`. Điểm trên biểu đồ generation là EX nội bộ của dự án.


In [ ]:
from IPython.display import Image, display
run("-m", "evaluation.plot_schema_retrieval",
    "--input", "docs/bird_minidev_retrieval_results.json",
    "--input", "docs/bird_minidev_dense_results.json",
    "--output", "results/plots/retrieval_reference", "--pdf")
display(Image(filename="results/plots/retrieval_reference/coverage_vs_tokens.png"))


In [ ]:
if RUN_GENERATION:
    for repeat in range(REPEATS):
        repeat_root = GENERATION_ROOT / f"repeat_{repeat:02d}"
        compare_args = ["-m", "evaluation.compare_schema_runs"]
        for mode in ARMS:  # full luôn đứng đầu lệnh compare, bất kể thứ tự thực thi.
            compare_args += ["--run", f"{mode}={repeat_root / mode}"]
        comparison = WORK / "results/plots" / f"repeat_{repeat:02d}"
        run(*compare_args, "--output", comparison, "--bootstrap-unit", "question", "--pdf")
        table = pd.read_csv(comparison / "comparison.csv")
        display(table[["run", "questions", "project_ex_pct", "ex_delta_pp", "gained", "lost",
                       "input_tokens_first_reduction_pct", "end_to_end_speedup_paired"]])
        for filename in ["accuracy_and_transitions", "accuracy_delta_ci", "input_tokens_first", "elapsed_ms",
                         "paired_latency", "cuda_peak_allocated_mb_device_max", "error_breakdown"]:
            image = comparison / f"{filename}.png"
            if image.exists():
                display(Image(filename=str(image)))


## 6. Chạy BIRD EX chính thức (sau full run)

Gắn source evaluator chính thức tại commit `abd11b6db92a1c9f809b32f7564c7c71b34d67f0`, sửa đường dẫn dưới. Exporter dùng đúng subset và giữ thứ tự. Scorer Mini-Dev cần đủ ba nhóm difficulty; dùng full run thay vì smoke test. Lệnh này không đo inference latency hay VRAM.


In [ ]:
RUN_OFFICIAL_EX = False
OFFICIAL_EVALUATOR = Path("/kaggle/input/bird-mini-dev-source/mini_dev/evaluation/evaluation_ex.py")
if RUN_GENERATION and RUN_OFFICIAL_EX:
    assert LIMIT is None, "Chạy đủ tập trước khi dùng scorer có breakdown ba difficulty"
    assert OFFICIAL_EVALUATOR.is_file(), "Sửa đường dẫn source scorer chính thức"
    run("-m", "pip", "install", "func-timeout==4.3.5", "psycopg2-binary==2.9.9", "PyMySQL==1.1.1")
    for repeat in range(REPEATS):
        for mode in ARMS:
            arm_dir = GENERATION_ROOT / f"repeat_{repeat:02d}" / mode
            official = arm_dir / "official"
            run("scripts/export_bird_predictions.py", "--dataset", EVAL_DATASET,
                "--predictions", arm_dir / "predictions.jsonl", "--output", official)
            # Scorer append log; xóa log của lần chạy cũ cùng thư mục.
            (official / "ex.txt").unlink(missing_ok=True)
            run(OFFICIAL_EVALUATOR, "--predicted_sql_path", official / "predict_mini_dev.json",
                "--ground_truth_path", official / "mini_dev_gold.sql",
                "--db_root_path", str(DB_ROOT.resolve()) + "/",
                "--diff_json_path", official / "mini_dev_difficulty.jsonl",
                "--sql_dialect", "SQLite", "--num_cpus", "2", "--meta_time_out", "30.0",
                "--output_log_path", official / "ex.txt")


## 7. Lưu kết quả và kiểm tra scorer chính thức

`summary.json` chứa thống kê, `predictions.jsonl` chứa từng câu với SQL chọn, lỗi, input/output token và peak CUDA memory. Retrieval có `details.jsonl` để xem bảng/cột bị mất. Giữ cả failed/context overflow trong mẫu số.

Bộ so sánh execution nội bộ giữ duplicate và xét ORDER BY/LIMIT; nó không hoàn toàn giống scorer BIRD. Trước khi công bố điểm benchmark, xuất prediction theo hướng dẫn evaluator chính thức ở [BIRD Mini-Dev](https://github.com/bird-bench/mini_dev), chạy đúng SQLite/GT cùng phiên bản. Không dùng kết quả proxy token hoặc mock để báo EX/VES.

Exporter có sẵn: `python scripts/export_bird_predictions.py --dataset <subset-evaluated.json> --predictions <predictions.jsonl> --output <official-dir>`. Xem hướng dẫn `docs/SCHEMA_PRUNING_BIRD_KAGGLE.md` cho lệnh EX đã ghim commit, JSONL difficulty và yêu cầu DB cùng phiên bản.


In [ ]:
archive = shutil.make_archive("/kaggle/working/schema_pruning_results", "zip", WORK / "results")
print(archive)
